# 02. Advanced Statistical Analysis & Econometric Panel Regression
**Project:** Downtown Seattle Economic Vitality & MID Operations (2025–2026)
**Author:** Daniel Rodriguez III

This notebook authenticates with Google Cloud, retrieves the denormalized monthly analytical view, and runs robust econometric models (OLS with HC3 Heteroskedasticity-Consistent Standard Errors) to validate the operational impact of MID tickets on pedestrian volume recovery.

In [ ]:
from google.cloud import bigquery
from google.colab import auth
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Authenticate
print("Requesting credentials...")
auth.authenticate_user()
client = bigquery.Client(project='driiiportfolio')

# 2. Extract
query = """
SELECT * 
FROM `driiiportfolio.dsa_urban_analytics.v_monthly_downtown_dashboard`
WHERE month_start >= '2025-01-01'
"""
df_analysis = client.query(query).to_dataframe()
print(f"Loaded {len(df_analysis)} analytical rows.")

In [ ]:
# 3. Pearson Correlation Analysis
corr_coef, p_value = stats.pearsonr(df_analysis['total_mid_tickets'], df_analysis['total_foot_traffic'])
print(f"Pearson Correlation: {corr_coef:.3f} (p-value: {p_value:.4f})")

# 4. Econometric Regression with HC3 Robust Standard Errors
model = smf.ols("total_foot_traffic ~ total_mid_tickets + avg_ops_response_time_min", data=df_analysis).fit(cov_type='HC3')
print(model.summary())

In [ ]:
# 5. Visual Validation Plotting
fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(data=df_analysis, x='total_mid_tickets', y='total_foot_traffic', ax=ax, line_kws={'color': 'red'})
ax.set_title('MID Service Deployments vs. Pedestrian Foot Traffic Volume')
plt.show()